# LoRa-Modulation
**Procesamiento Estocástico de Señales** — FCEFyN, UNC

Modulación tipo LoRa (CSS, *Chirp Spread Spectrum*) en Python y prueba por aire con un ADALM-Pluto.
Referencia: L. Vangelista, *"Frequency Shift Chirp Modulation: The LoRa Modulation"*, IEEE SPL, vol. 24, n.º 12, 2017.

Las celdas `%%writefile` guardan cada módulo del proyecto en un `.py` tal cual está en el repositorio, y después se importan.

## 0. Requisitos
`numpy`, `matplotlib`, `scipy` y, solo para la parte de hardware, `pyadi-iio`. Descomentar la línea si falta algo.

In [ ]:
# %pip install numpy matplotlib scipy pyadi-iio
%matplotlib inline

## 1. Capa 1 — Generación de señal (`Chirp.py`)
`ChirpFunction` genera el chirp ideal de `2^SF` muestras; `NoisyChirp` le suma ruido gaussiano complejo.
Las funciones `chirpSeq`, `makeChirpPreamble`, `makeFastChirp`, `chirpFactory` y `makeSilence` entregan `SamplesAndTime`.

In [ ]:
%%writefile Chirp.py
import SDR as SDR
import numpy as np
import matplotlib as plt



### Make an array of chirp (same simbol)
def makeChirpPreamble(SF, noise):
    M = 2 ** SF * ChirpFunction.resolution

    ups = chirpSeq(SF, [0] * 8, noise, 1.0, 0, 0)


    downs = chirpSeq(SF, [0] * 3, noise, -1.0, 0, 0)
    downs.trunk(3 * M // 4)

    samples = np.concatenate([ups.samples, downs.samples])
    time = np.arange(len(samples)) / ChirpFunction.resolution
    return SDR.SamplesAndTime(time, samples)

def chirpSeq(SF, symbols, noise, up, startDelay, endDelay):
    M = 2 ** SF * ChirpFunction.resolution  # samples per symbol

    chunks = []
    for s in symbols:
        chirp = makeFastChirp(SF, s, noise, up)
        chunks.append(chirp.samples)
    samples = np.concatenate(chunks)

    # Delays are given in symbols, converted to samples here
    start = makeSilence(startDelay * M, noise)
    end = makeSilence(endDelay * M, noise)
    samples = np.concatenate([start, samples, end])

    time = np.arange(len(samples)) / ChirpFunction.resolution
    return SDR.SamplesAndTime(time, samples)


def makeSilence(length, noise):
    return noise * (np.random.randn(length) + 1j * np.random.randn(length))

def chirpFactory(N, SF, symbol, noise, up):
    functions = []
    for _ in range(N):
        functions.append(makeFastChirp(SF, symbol, noise, up))
    return functions


def makeFastChirp(SF, symbol, noise, up):
    a = ChirpFunction(SF, symbol, up)
    b = NoisyChirp(a, noise)
    return SDR.SamplesAndTime(b.time, b.getSamples())

class ChirpFunction:
    resolution = 1
    def __init__(self,SF:int,symbol:int,up):
        self.SF = SF
        self.up = up
        self.symbol = symbol
        (self.k,self.f) = self.makeF()

    def graphReal(self):
        plt.plot(self.k,self.f.real)
        plt.show()
    def graphPhase(self):
        plt.plot(self.k, self.phase)
        plt.show()
    def graphFrec(self):
        plt.plot(self.k, self.frec)
        plt.show()
    def makeF(self):
            N = 2 ** self.SF
            t = np.arange(N * self.resolution) / self.resolution
            phase = 2 * np.pi * ((self.symbol + t) % N) * t / N
            f = np.exp(1j * self.up * phase) / np.sqrt(N)
            return t, f

class NoisyChirp:
        def __init__(self, chirpFunction, noise):
            self.chirpFunction = chirpFunction
            self.time = chirpFunction.k.copy()
            n = len(chirpFunction.f)
            ruido = noise * (np.random.randn(n) + 1j * np.random.randn(n))
            self.samples = chirpFunction.f + ruido
            self.resolution = chirpFunction.resolution

        def getTime(self): return self.time

        def getSamples(self): return self.samples

## 2. Capa 2 — Procesamiento (`SDR.py` → `SamplesAndTime`)
Array complejo + eje de tiempo + operaciones DSP: dechirp + FFT (`chirpDown`), estimación de símbolo, detección de preámbulo por correlación y gráficos.

In [ ]:
%%writefile SDR.py
from json.decoder import NaN

import numpy as np
import matplotlib.pyplot as plt
from scipy.stats import false_discovery_control




class  SamplesAndTime:
    def __init__(self,time,samples):
        #init time & samples
        self.time = time
        self.samples = samples
        #N the number of samples on a chirp is the SF
        self.N = len(samples)
        self.chirpDown()

    def graphSpectrogram(self, fs=None, nfft=32, ax=None, title=None):
        # Simple spectrogram: time on x, frequency on y, color is power (dB)
        show = ax is None
        if ax is None:
            ax = plt.gca()

        x = np.asarray(self.samples)
        nfft = min(nfft, len(x))
        # fs=None -> normalized frequency (cycles/sample)
        ax.specgram(x, NFFT=nfft, Fs=fs if fs else 1.0,
                    noverlap=nfft * 3 // 4, cmap="viridis")

        ax.set_xlabel("Time (s)" if fs else "Time (samples)")
        ax.set_ylabel("Frequency (Hz)" if fs else "Frequency (cycles/sample)")
        if title:
            ax.set_title(title)
        if show:
            plt.show()


    def trunk(self, trunk):
        n = len(self.time) - trunk
        self.time = self.time[:n]
        self.samples = self.samples[:n]
        self.N = n
        self.chirpDown()

    def graphFrequencyChirp(self, fs=None, ax=None, title=None):
        z = self.samples[1:] * np.conj(self.samples[:-1])
        dt = np.diff(self.time)
        frequency = np.angle(z) / (2 * np.pi * dt)
        frequency[np.abs(z) == 0] = 0
        time_mid = self.time[:-1] + dt / 2

        if fs is not None:
            frequency = frequency * fs / 1e6  # MHz
            time_mid = time_mid / fs * 1e6  # us
            xlabel, ylabel = "Time (us)", "Frequency (MHz)"
        else:
            xlabel, ylabel = "Time", "Frequency"

        show = ax is None
        if ax is None:
            ax = plt.gca()
        ax.plot(time_mid, frequency, ".-", markersize=3)
        ax.set_xlabel(xlabel)
        ax.set_ylabel(ylabel)
        ax.grid(True)
        if title:
            ax.set_title(title)
        if show:
            plt.show()

    def SamplesAndTime(self):
        plt.plot(self.time, self.samples.real)
        plt.show()
    def graphReal(self):
        plt.plot(self.time, self.samples.real)
        plt.show()
    def chirpDown(self):
        d = self.samples * np.exp(-2j * np.pi * (self.time ** 2) / self.N)
        self.fft = np.fft.fft(d) / np.sqrt(self.N)
    def graphFFT(self):
        plt.plot(self.time, abs((self.fft)))
        plt.show()
    def estimateSymbol(self):
        return np.argmax(np.abs(self.fft))
    def isSymbolCorrect(self,symbol):
        return symbol == self.estimateSymbol()

    def graphFrequency(self):
        z = self.samples[1:] * np.conj(self.samples[:-1])
        dt = np.diff(self.time)
        frequency = np.angle(z) / (2 * np.pi * dt)
        frequency[np.abs(z) == 0] = 0   # silence drawn as a flat line at 0
        time_mid = self.time[:-1] + dt / 2
        plt.plot(time_mid, frequency)
        plt.xlabel("Time")
        plt.ylabel("Frequency")
        plt.grid(True)
        plt.show()
    def displaceTime(self,displacement):
        self.k = self.k+displacement

    def getPreamble(self, preamble, threshold_ratio=0.8):
        p = np.asarray(getattr(preamble, "samples", preamble))
        c = np.abs(np.correlate(self.samples, p, mode="full"))

        dt = self.time[1] - self.time[0]
        lags = np.arange(-(len(p) - 1), len(self.samples)) * dt

        threshold = threshold_ratio * np.sum(np.abs(p) ** 2)
        idx = np.argmax(c)
        if c[idx] > threshold:
            return True, lags[idx]
        return False, np.nan

    def getPreambleEnergy(self, preamble, threshold_ratio=0.8):
        p = np.asarray(getattr(preamble, "samples", preamble))
        x = self.samples


        c = np.abs(np.correlate(x, p, mode="valid"))


        win_energy = np.convolve(np.abs(x) ** 2, np.ones(len(p)), mode="valid")
        norm = np.sqrt(win_energy * np.sum(np.abs(p) ** 2))
        c = c / np.where(norm == 0, np.inf, norm)

        dt = self.time[1] - self.time[0]
        idx = np.argmax(c)
        if c[idx] > threshold_ratio:
            return True, idx * dt
        return False, np.nan


    def graphCorrelationPreamble(self, preamble):
        p = np.asarray(getattr(preamble, "samples", preamble))
        c = np.correlate(self.samples, p, mode="full")

        dt = self.time[1] - self.time[0]
        lags = np.arange(-(len(p) - 1), len(self.samples)) * dt

        idx = np.argmax(np.abs(c))
        print(f"Máximo: {np.abs(c)[idx]:.2f} en desfase {lags[idx]:.2f}")

        plt.plot(lags, np.abs(c))
        plt.xlabel("desfase")
        plt.ylabel("|correlación|")
        plt.grid(True)
        plt.show()
        return lags, c

## 3. Capa 3 — Hardware (`PlutoSDR.py`)
`PlutoTx` y `PlutoRx` controlan el ADALM-Pluto sin conocer la señal que transmiten o reciben.

In [ ]:
%%writefile PlutoSDR.py
import numpy as np

SCALE = 2**13 * np.sqrt(2**8)


class Pluto:
    def __init__(self, sdr, bw, lo, SF):
        self.sdr = sdr
        self.uri = sdr.uri      # ip comes from the sdr object
        self.bw = bw
        self.lo = lo
        self.SF = SF
        self.sdr.sample_rate = bw


class PlutoTx(Pluto):
    def __init__(self, sdr, bw, gain, SF, lo):
        super().__init__(sdr, bw, lo, SF)
        self.gain = gain                      # range: -89.75 to 0 dB
        self.sdr.tx_lo = lo
        self.sdr.tx_rf_bandwidth = bw
        self.sdr.tx_hardwaregain_chan0 = gain

    def send(self, samples, cyclic=True):
        # WARNING: with cyclic=True the buffer transmits forever.
        # Call stop() when done, otherwise the Pluto keeps emitting.
        self.sdr.tx_cyclic_buffer = cyclic    # must be set before tx()
        self.sdr.tx(np.asarray(samples) * SCALE)

    def sendCyclic(self, samples):
        # Workaround: the Pluto only repeats the first quarter of the buffer
        self.send(np.tile(samples, 4))

    def stop(self):
        # Stop the cyclic transmission
        self.sdr.tx_destroy_buffer()


class PlutoRx(Pluto):
    def __init__(self, sdr, bw, gain, SF, lo, buffer_size=2**11):
        super().__init__(sdr, bw, lo, SF)
        self.gain = gain                      # range: 0 to 73 dB
        self.buffer_size = buffer_size
        self.sdr.rx_lo = lo
        self.sdr.rx_rf_bandwidth = bw
        self.sdr.gain_control_mode_chan0 = "manual"
        self.sdr.rx_hardwaregain_chan0 = gain
        self.sdr.rx_buffer_size = buffer_size

    def receive(self):
        self.sdr.rx()                         # discard first buffer
        return self.sdr.rx()

    def listen(self, detector):
        # detector(rx_samples) -> (found, lag). The Pluto doesn't know
        # what is being detected; whoever calls decides.
        self.sdr.rx()  # discard first buffer

        # Block until the detector fires

        while True:
            rx_samples = self.sdr.rx()
            found, lag = detector(rx_samples)
            print("listing preamble...")
            if found:
                return rx_samples, lag

    def stop(self):
        self.sdr.rx_destroy_buffer()

## 4. Orquestador (`testerLab.py` → `RealTester`)
Única parte que conoce las tres capas.

In [ ]:
%%writefile testerLab.py
import adi
import numpy as np

import Chirp
import SDR
from PlutoSDR import PlutoTx, PlutoRx


class RealTester:
    def __init__(self, SF=8, frec=int(2.4e9), bw=int(1e6), uri="ip:192.168.1.35"):
        self.SF = SF
        self.frec = frec
        self.bw = bw
        self.uri = uri

    # ---------- helpers (lo que antes vivía en PlutoTx / PlutoRx) ----------
    def _sendChirp(self, tx, symbols, up=1.0, noise=0.0, startDelay=0, endDelay=0):
        data = Chirp.chirpSeq(self.SF, symbols, noise, up, startDelay, endDelay)
        tx.sendCyclic(data.samples)
        return data

    def _sendPreamble(self, tx, noise=0.0):
        data = Chirp.makeChirpPreamble(self.SF, noise)
        data.graphSpectrogram(title="Theorical")
        tx.sendCyclic(data.samples)
        return data

    def _preambleDetector(self, threshold_ratio, preamble=None):
        if preamble is None:
            preamble = Chirp.makeChirpPreamble(self.SF, 0.0)

        def detector(rx_samples):
            time = np.arange(len(rx_samples))
            st = SDR.SamplesAndTime(time, rx_samples)
            return st.getPreambleEnergy(preamble, threshold_ratio)

        return detector

    # ---------- tests ----------
    def preambleTest(self, tx_gain=-10, rx_gain=40, threshold_ratio=0.95,
                     buffer_size=2 ** 15, window=3000):
        # One Pluto shared by TX and RX (loopback)
        dev = adi.Pluto(uri=self.uri)
        tx = PlutoTx(dev, self.bw, tx_gain, self.SF, self.frec)
        rx = PlutoRx(dev, self.bw, rx_gain, self.SF, self.frec, buffer_size=buffer_size)

        try:
            # Transmitter: preamble repeats until stop()
            self._sendPreamble(tx)

            # Receiver: blocks until the preamble is detected
            rx_samples, lag = rx.listen(self._preambleDetector(threshold_ratio))
            samples = SDR.SamplesAndTime(np.arange(window), rx_samples[:window])
            samples.graphSpectrogram(nfft=16, title="Received preamble")
            print(f"Preamble found at sample {lag}")
        finally:
            # Always stop the cyclic transmission
            tx.stop()
            rx.stop()

        # Plot what was received
        st = SDR.SamplesAndTime(np.arange(len(rx_samples)), rx_samples)
        st.graphSpectrogram(nfft=16, title="Received preamble")
        st.graphFrequencyChirp(title="Instantaneous frequency")

    def graphChirp(self, tx_gain=0, rx_gain=40, buffer_size=2 ** 13, window=2800):
        dev = adi.Pluto(uri=self.uri)
        rx = PlutoRx(dev, self.bw, rx_gain, self.SF, self.frec, buffer_size=buffer_size)
        tx = PlutoTx(dev, self.bw, tx_gain, self.SF, self.frec)
        # transmit
        self._sendPreamble(tx)
        # receive
        rx.sdr.rx()
        samples = rx.sdr.rx()
        SaT = SDR.SamplesAndTime(np.arange(window), samples[:window])
        SaT.graphSpectrogram(nfft=128, title="Received preamble")


if __name__ == "__main__":
    RealTester().preambleTest(tx_gain=-65,threshold_ratio=0.6)

## 5. Importar los módulos

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

import Chirp
import SDR

## 6. Simulación — modulación y demodulación
Se modula una secuencia sin ruido y se demodula cortando en bloques de `2^SF` muestras.

In [ ]:
SF = 5
M = 2 ** SF
symbols = [0, 22, 31, 21]

# Capa 1: generar la señal
seq = Chirp.chirpSeq(SF, symbols, noise=0.0, up=1.0, startDelay=0, endDelay=0)

# Capa 2: cortar en bloques de 2^SF y demodular
decoded = []
for i in range(len(symbols)):
    block = seq.samples[i * M:(i + 1) * M]
    decoded.append(int(SDR.SamplesAndTime(np.arange(M), block).estimateSymbol()))

print(decoded)  # [0, 22, 31, 21]

## 7. Simulación — detección de preámbulo en un buffer largo

In [ ]:
SF = 8
M = 2 ** SF
noise = 0.05

preamble = Chirp.makeChirpPreamble(SF, 0.0)        # referencia sin ruido
rx_pre   = Chirp.makeChirpPreamble(SF, noise)      # lo que "llega", con ruido
silence  = Chirp.makeSilence(3 * M, noise)

rx = np.concatenate([silence, rx_pre.samples, silence])
st = SDR.SamplesAndTime(np.arange(len(rx)), rx)

found, lag = st.getPreambleEnergy(preamble, threshold_ratio=0.5)
print(found, lag)  # True, y el desfase debería caer en 3 * M

## 8. Prueba por aire — preámbulo real (requiere ADALM-Pluto)
Equivale a `uv run src/testerLab.py`. Antes de correr, revisar la `uri` del Pluto en `RealTester` (por defecto `ip:192.168.1.35`).

⚠️ El preámbulo se transmite en forma cíclica; `preambleTest` corta la transmisión en el `finally`.

In [ ]:
from testerLab import RealTester

RealTester().preambleTest(tx_gain=-65, threshold_ratio=0.6)